# HACS208c COMPAS
This notebook provides a template to train your own recidivism risk prediction model.

## Background
We will use the parsed FOIA'd dataset from the analysis completed by ProPublica exposing bias in the COMPAS recidivism risk analysis model. The results of the original analysis are published in this news article: https://www.propublica.org/article/machine-bias-risk-assessments-in-criminal-sentencing. As an example of the impact of undefined and non-transparent definitions of fairness and bias, Northpointe (creators of COMPAS) has their rebuttal available here: https://go.volarisgroup.com/rs/430-MBX-989/images/ProPublica_Commentary_Final_070616.pdf (It is still available via the Wayback Machine). 

## Instructions
The original data has been split into a `train.csv` and `test.csv`. Only use the `train.csv` during model training. 

Play around with __feature engineering, model selection, and objective functions__ to try to build a __harmless__ model when predicting whether or not an individual will commit `is_violent_recid`

Compare your results with the original COMPAS model

Be prepared to share your findings on how unbiased/biased harmless/harmful your model is and how you arrived at model decisions. 

## Steps are as follows:
1. Setup and basic data cleaning (I generally do not recommend changing this)
1. Feature Engineering
1. Model and Objective Selection
1. Model Training
1. Evaluate on Test Data
    - (Optional) Compare via Model Facts Label

## Important Columns
- `v_score_text`: The COMPAS model score label (Low, Medium, High) from their violent recidivism prediction model
- `v_decile_score`: The COMPAS quantitative score (1-10)
- `is_violent_recid`: Whether or not they commited a new violent crime within two years after release (0 = not a recidivist)
- `priors_count`: The number of prior crimes. For the sake of the Model Facts Label, we will use this as a baseline for comparison

# 1. Setup
- Install necessary packages and load necessary functions
- Download the `train.csv` dataset 
- Clean data per Propublica methodology

##  Data Cleaning
- Filters out bad data:
    - Some of the data is missing scores or have start dates before end dates, demonstrating potential errors in the data. This is in line with the Propublica team's method in https://github.com/propublica/compas-analysis/blob/master/Compas%20Analysis.ipynb
- For comparison with the original model: Convert COMPAS labels (for accuracy calculations): we label all data with "High" COMPAS labels to be a prediction of violent recidivism. "Medium" and "Low" are set to 0. 
    - The Propublica team labeled High and Medium as 1. However, the Northpointe team took issue with this in their rebuttal
    - Saved in `COMPAS_pred` column
- Create a baseline prediction: While no baseline model is available, we create our own baseline under the assumption that people with a greater criminal history (`prior_counts`) are more likely to be a violent recidivist. 
    - We establish a threshold to mirror the percentage of violent recidivists in the data. 
    - This baseline should not be used in real life. It propagates the systemic biases caused by racism and overpolicing of Black and Brown neighborhoods.
    - Binary predictions saved to `BASELINE_pred` column; Probabilities saved in `BASELINE_proba`
- Format data for easy Model Facts label generation 

In [ ]:
# Download packages 
!pip install modelfacts, ydata-profiling, 'setuptools<=80.10.2'

In [2]:
# Import packages and some initial prep functions (DO NOT CHANGE unless you know what you are doing)
import os
from shlex import split
import requests
import pandas as pd
import datetime as dt
from sklearn import metrics

from modelfacts import  ModelFacts


def download_data(url: str, save_path: str) -> None:
    """
    Downloads a file from the specified URL and saves it to the given path.
    
    Args:
        url (str): The URL of the file to download.
        save_path (str): The local path where the file should be saved.
    """
    print(f"Downloading file from {url}...")
    response = requests.get(url)

    if response.status_code == 200:
        # Ensure the directory exists
        os.makedirs(os.path.dirname(save_path) if os.path.dirname(save_path) else '.', exist_ok=True)
        
        # Write the downloaded content
        with open(save_path, 'wb') as f:
            f.write(response.content)
        print(f"Successfully saved to: {save_path}")
    else:
        print(f"Failed to download. Status code: {response.status_code}")

def prep_data(data: pd.DataFrame) -> pd.DataFrame:
    """
    Prepares the data by filtering and creating baseline and prediction columns.
    
    Args:
        data (pd.DataFrame): The input DataFrame to prepare.
    """
    # get rid of missing COMPAS values and invalid rows
    data = data[(~data['score_text'].isna()) & (data['end']>data['start'])].copy()
    # create baseline for comparison based on priors count 
    # the cutoff is the linear quantile of the percent of recidivists
    split = {'High': 1, 'Medium':0, 'Low':0}
    perc_recid = sum(data['is_violent_recid'])/len(data)
    cutoff = data['priors_count'].quantile(q=1-perc_recid)
    baseline_text = data['priors_count'].where(data['priors_count']>0, 'Low')\
        .where((data['priors_count']<=0) | (data['priors_count']>cutoff), 'Medium')\
        .where(data['priors_count']<=cutoff, 'High')

    data['BASELINE_proba'] = data['priors_count']
    data['BASELINE_pred'] = baseline_text.replace(split)
    # create prediction column based on COMPAS score_text
    data['COMPAS_pred'] = data['v_score_text'].replace(split)
    return data

In [3]:
# set data path 
train_path = "new_data/train.csv"
test_path = "new_data/test.csv"

In [4]:
# download train and test data

TRAIN_GITHUB_URL = 'https://raw.githubusercontent.com/jhzsquared/hacs208c_lecture/main/data/train.csv'
TEST_GITHUB_URL = 'https://raw.githubusercontent.com/jhzsquared/hacs208c_lecture/main/data/test.csv'

download_data(TRAIN_GITHUB_URL, train_path)
download_data(TEST_GITHUB_URL, test_path)

Successfully saved to: new_data/train.csv
Successfully saved to: new_data/test.csv


In [5]:
# load in training data
train_data = prep_data(pd.read_csv(train_path))
print("Size of training data (rows, columns):", train_data.shape)
train_data.head()

Size of training data (rows, columns): (14540, 55)


/tmp/ipykernel_29139/1261523218.py:53: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['BASELINE_pred'] = baseline_text.replace(split)
/tmp/ipykernel_29139/1261523218.py:55: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['COMPAS_pred'] = data['v_score_text'].replace(split)


,id,name,first,last,compas_screening_date,sex,dob,age,age_cat,race,...,v_screening_date,in_custody,out_custody,priors_count.1,start,end,event,BASELINE_proba,BASELINE_pred,COMPAS_pred
0,1,miguel hernandez,miguel,hernandez,2013-08-14,Male,1947-04-18,69,Greater than 45,Other,...,2013-08-14,2014-07-07,2014-07-14,0,334,961,0,0,0,0
1,2,michael ryan,michael,ryan,2014-12-31,Male,1985-02-06,31,25 - 45,Caucasian,...,2014-12-31,2014-12-30,2015-01-03,0,3,457,0,0,0,0
2,4,ed philo,ed,philo,2013-04-14,Male,1991-05-14,24,Less than 25,African-American,...,2013-04-14,2013-06-16,2013-06-16,4,0,63,0,4,0,0
3,4,ed philo,ed,philo,2013-04-14,Male,1991-05-14,24,Less than 25,African-American,...,2013-04-14,2013-07-30,2013-11-08,4,63,107,0,4,0,0
4,4,ed philo,ed,philo,2013-04-14,Male,1991-05-14,24,Less than 25,African-American,...,2013-04-14,2014-03-27,2014-05-02,4,208,347,0,4,0,0


# 2. Feature Engineering
Explore the data, decide what features you want to use to train predictions, and how you want to transform them

In [8]:
# Explore the data (I like to use this package but feel free to make your own)
from ydata_profiling import ProfileReport

profile = ProfileReport(train_data, title="Data Exploration Report")
profile

/tmp/ipykernel_29139/2452029618.py:2: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 55/55 [00:00<00:00, 321.01it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# Transform data as needed. Save them as new columns in the dataframe.
# (e.g. log transform, normalize, or one hot encode)
# Hot tip: Wrap it in a function or Sklearn ColumnTransformer since you'll need to do it again on the test data later!

# columns_to_transform = ["INSERT COLUMN NAMES"]
# INSERT TRANSFORMATION CODE HERE

In [ ]:
# Save out columns you want to train your model on
# `is_recid` is the target variable, so DO NOT include it in this list

columns_to_use = ["INSERT COLUMN NAMES"]

In [ ]:
columns_to_use = ["sex", "age", "race", "juv_fel_count", "juv_misd_count", "juv_other_count",]

# 3. Model and Objective Selection
For simplicity, I recommend using the scikit-learn package's classification models 
For more information and examples: https://scikit-learn.org/stable/auto_examples/classification/plot_classifier_comparison.html 

Example if you decided on using a logistic regression:
```


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline


X = train_data[columns_to_use]
y = train_data["is_recid"]

model = Pipeline([
    ("classifier", LogisticRegression(max_iter=1000)),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](3,)","[-1, 0, 1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](6,)","['sex','age','race','juv_fel_count','juv_misd_count','juv_other_count']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,6
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='

In [12]:

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))




              precision    recall  f1-score   support

          -1       0.00      0.00      0.00       133
           0       0.58      0.68      0.62      1449
           1       0.58      0.52      0.55      1326

    accuracy                           0.58      2908
   macro avg       0.38      0.40      0.39      2908
weighted avg       0.55      0.58      0.56      2908



/home/jessz/umd/research/hacs208c/.hacs208c/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/jessz/umd/research/hacs208c/.hacs208c/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/jessz/umd/research/hacs208c/.hacs208c/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

# 4. Evaluate on Test Data

Obtain the predictions on the test data and evaluate it for harms

Come up with your own evaluation method, make some charts, and/or make  a Model Facts Label

In [21]:
# load in test data
test_data = prep_data(pd.read_csv(test_path))
print("Size of test data (rows, columns):", test_data.shape)
test_data.head()

Size of test data (rows, columns): (3638, 55)


,id,name,first,last,compas_screening_date,sex,dob,age,age_cat,race,...,v_screening_date,in_custody,out_custody,priors_count.1,start,end,event,BASELINE_proba,BASELINE_pred,COMPAS_pred
0,4958,dylan welly,dylan,welly,2013-03-16,Male,1994-04-18,22,Less than 25,Caucasian,...,2013-03-16,2013-10-04,2013-10-09,0,207,1112,0,0,0,0
1,4173,guillermo laffiteau,guillermo,laffiteau,2014-06-20,Male,1957-05-23,58,Greater than 45,Hispanic,...,2014-06-20,2014-06-19,2014-06-21,4,1,651,0,4,0,0
2,5898,luis gonzalez,luis,gonzalez,2013-12-26,Male,1992-05-22,23,Less than 25,Caucasian,...,2013-12-26,2013-12-25,2013-12-26,0,0,827,0,0,0,0
3,8952,joseph ambers,joseph,ambers,2014-08-02,Male,1966-09-17,49,Greater than 45,Caucasian,...,2014-08-02,2016-03-10,2020-01-01,6,608,609,0,6,0,0
4,9052,anthony nicholasi,anthony,nicholasi,2014-07-01,Male,1993-11-15,22,Less than 25,Caucasian,...,2014-07-01,2014-06-30,2014-07-01,1,0,640,0,1,0,0


## Create Model Facts Label
Generate a Model Facts Label based on your model's predictions. 
Update the administrative details as needed (currently set with default values)

Variables in all caps likely do not need to be adjusted, though you can set the baseline as the Northpointe model if desired. Variables in lower case do need to be updated with your model's information

In [ ]:
# Save your model's predictions to a new column for Model Facts Label


In [25]:
# set admin details (you should not need to change these)
APPLICATION = "Predicting risk of violent recidivism using COMPAS. " \
"The target class is predicting violent recidivism"
MODEL_TYPE = "classification"
WARNINGS = "This model has been demonstrated to propagate biases by ProPublica." \
 " Its creators claim this model is unbiased, under the predictive parity paradigm using AUC." \
 " Without a clear definition of fairness, it should not be used in decision making"
SOURCE = "Data from Broward County, Florida https://github.com/propublica/compas-analysis/tree/master." \
" Model created by Northpointe"
TRAIN_DATE = max(pd.to_datetime(test_data['compas_screening_date'])) # date compas model was used
TEST_DATA_DATE = dt.datetime(2016,5,23) # propublica article publication
DATA_SPLIT = "80/20" # train/test split
DATA_SIZE = len(test_data)+len(train_data)
AGE_COL = "age" # which column has age information in it
DEMO_COLS = ['race', 'sex', 'age'] # which columns has additional demographic information in it
TRUE_COL = "is_violent_recid"


In [ ]:
# Set your model specific information and create a Model Facts Label

# Baseline your model against these predictions
baseline_col = "BASELINE_pred" # could also be "COMPAS_pred" if you wanted to baseline it against COMPAS
baseline_proba = "BASELINE_proba"

# Update with your model's prediction columns 
pred_col = "COMPAS_pred"
pred_proba = "v_decile_score"

# define scoring functions
st_score = 'f1_score' # name of the standard scoring function applicable to this problem
st_kwargs = {} # e.g. {'average':'macro'} any of the keyword arguments for the score of interest
t_score = 'roc_auc_score' # name of the scoring function used for training optimization
t_kwargs = {} # same usage as st kwargs
t_score_func = getattr(metrics, t_score)
st_score_func = getattr(metrics, st_score)

model_facts = ModelFacts(test_data, TRUE_COL, pred_col, baseline_col, 
                         st_score_func, t_score_func, classification = True, 
                         pred_proba = pred_proba, baseline_proba = baseline_proba,
                         t_proba = False)
# calculate various stats along demographics
mf_compas = model_facts(DEMO_COLS, age_col = AGE_COL,
    train_date = TRAIN_DATE, test_data_date = TEST_DATA_DATE,
    data_size = DATA_SIZE, data_split = DATA_SPLIT,
    st_kwargs = st_kwargs, t_kwargs = t_kwargs)
# create the Model Facts label from the data
table = model_facts.make_label(mf_compas, APPLICATION, WARNINGS, SOURCE, show = True)

Model Facts 
 
 
 Application: Predicting risk of violent recidivism using COMPAS. The target class is predicting violent recidivism 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Model Type 
 classification 
 
 
 
 
 Model Train Date 
 31 December 2014 
 
 
 
 
 Test Data Date 
 23 May 2016 
 
 
 
 
 Dataset Size 
 18178 
 
 
 
 
 %Train/%Test 
 80/20 
 
 
 
 
 Accuracy 
 
 
 
 Name 
 Raw Score 
 % Over Baseline 
 
 
 Standard Score 
 f1_score 
 0.195 
 102 
 
 
 Training Score 
 roc_auc_score 
 0.573 
 11.8 
 
 
 
 
 
 Demographics 
 % in Test Data 
 Standard Score 
 % Target 
 
 
 race 
 
 
 African-American 
 52.9 
 0.221 
 9.46 
 
 
 Asian 
 0.247 
 0.00 
 11.1 
 
 
 Caucasian 
 33.8 
 0.0741 
 5.53 
 
 
 Hispanic 
 7.64 
 0.00 
 3.60 
 
 
 Native American 
 0.357 
 0.500 
 7.69 
 
 
 Other 
 5.09 
 0.480 
 7.57 
 
 
 sex 
 
 
 Female 
 18.1 
 0.119 
 4.70 
 
 
 Male 
 81.9 
 0.203 
 8.22 
 
 
 age 
 
 
 18-24 
 23.6 
 0.257 
 9.55 
 
 
 25-34 
 38.8 
 0.170 
 8.50 
 
 
 35-49 
 23.6 
 0.101 
 6.42 
 
 
 50-64 
 12.7 
 0.0741 
 3.89 
 
 
 64+ 
 1.29 
 0.00 
 2.13 
 
 
 Warnings: This model has been demonstrated to propagate biases by ProPublica. Its creators claim this model is unbiased, under the predictive parity paradigm using AUC. Without a clear definition of fairness, it should not be used in decision making Data from Broward County, Florida https://github.com/propublica/compas-analysis/tree/master. Model created by Northpointe How to use Model Facts: The first section, "Application" through "Test Data Date" is to check that this model is relevant and timely for your goals. Use the accuracy "Standard Score" to compare it to other models. Use the demographic breakdown to check for biases in protected attributes (eg, if one race is underrepresented in the "% Test Data" or "% Target" or has a large difference in accuracy compared to the overall model's "Standard Score").

## Compare to Original Model
As demonstrated in Northpointe's rebuttal, the scoring metric you use can be quite effective in masking biases. 

This Model Facts Label uses AUC (as Northpointe does) and calculates the results on the test data. 

In [26]:
# Define variables
baseline_col = "BASELINE_pred" # could also be "COMPAS_pred" if you wanted to baseline it against COMPAS
baseline_proba = "BASELINE_proba"

# Update with your model's prediction columns 
pred_col = "COMPAS_pred"
pred_proba = "v_decile_score"

t_score = 'roc_auc_score'
st_score = 'roc_auc_score'
st_proba = True
t_proba = True
t_score_func = getattr(metrics, t_score)
st_score_func = getattr(metrics, st_score)

model_facts_NP = ModelFacts(test_data, TRUE_COL, pred_col, baseline_col, 
                         st_score_func, t_score_func, classification = True, 
                         pred_proba = pred_proba, baseline_proba = baseline_proba,
                         t_proba = False)
# calculate various stats along demographics
mf_compas = model_facts_NP(DEMO_COLS, age_col = AGE_COL,
    train_date = TRAIN_DATE, test_data_date = TEST_DATA_DATE,
    data_size = DATA_SIZE, data_split = DATA_SPLIT,
    st_kwargs = st_kwargs, t_kwargs = t_kwargs)
# create the Model Facts label from the data
table = model_facts_NP.make_label(mf_compas, APPLICATION, WARNINGS, SOURCE, show = True)

Model Facts 
 
 
 Application: Predicting risk of violent recidivism using COMPAS. The target class is predicting violent recidivism 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Model Type 
 classification 
 
 
 
 
 Model Train Date 
 31 December 2014 
 
 
 
 
 Test Data Date 
 23 May 2016 
 
 
 
 
 Dataset Size 
 18178 
 
 
 
 
 %Train/%Test 
 80/20 
 
 
 
 
 Accuracy 
 
 
 
 Name 
 Raw Score 
 % Over Baseline 
 
 
 Standard Score 
 roc_auc_score 
 0.573 
 11.8 
 
 
 Training Score 
 roc_auc_score 
 0.573 
 11.8 
 
 
 
 
 
 Demographics 
 % in Test Data 
 Standard Score 
 % Target 
 
 
 race 
 
 
 African-American 
 52.9 
 0.578 
 9.46 
 
 
 Asian 
 0.247 
 0.500 
 11.1 
 
 
 Caucasian 
 33.8 
 0.510 
 5.53 
 
 
 Hispanic 
 7.64 
 0.465 
 3.60 
 
 
 Native American 
 0.357 
 0.917 
 7.69 
 
 
 Other 
 5.09 
 0.700 
 7.57 
 
 
 sex 
 
 
 Female 
 18.1 
 0.539 
 4.70 
 
 
 Male 
 81.9 
 0.573 
 8.22 
 
 
 age 
 
 
 18-24 
 23.6 
 0.625 
 9.55 
 
 
 25-34 
 38.8 
 0.546 
 8.50 
 
 
 35-49 
 23.6 
 0.521 
 6.42 
 
 
 50-64 
 12.7 
 0.519 
 3.89 
 
 
 64+ 
 1.29 
 0.500 
 2.13 
 
 
 Warnings: This model has been demonstrated to propagate biases by ProPublica. Its creators claim this model is unbiased, under the predictive parity paradigm using AUC. Without a clear definition of fairness, it should not be used in decision making Data from Broward County, Florida https://github.com/propublica/compas-analysis/tree/master. Model created by Northpointe How to use Model Facts: The first section, "Application" through "Test Data Date" is to check that this model is relevant and timely for your goals. Use the accuracy "Standard Score" to compare it to other models. Use the demographic breakdown to check for biases in protected attributes (eg, if one race is underrepresented in the "% Test Data" or "% Target" or has a large difference in accuracy compared to the overall model's "Standard Score").

## Alternatively: Use Ipywidgets to build the label
Build Model Facts Label using a form 
You will need to save out your test data into a csv so that it can be uploaded

In [ ]:
# Save test data with predictions to a new csv file
pred_path = "new_data/test_with_predictions.csv"
test_data.to_csv(pred_path, index=False)

In [46]:
from modelfacts import ModelFactsWidget
wm = ModelFactsWidget()
wm

ModelFactsWidget(children=(Textarea(value='', description='What is the application of this model?', layout=Lay…

In [ ]:
# Run this cell after completing the form
# Creates the label 
# You can rerun this cell after reselecting values in the widget 
# without restarting the notebook
table = wm.create_label()